# Optimizerの比較とSchedulerによる学習率の調整

---
## 目的
代表的な最適化手法（Optimizer）であるSGD+Momentum，RMSprop，Adam，AdamWの特徴と挙動の違いを理解する．
また，学習の進行にあわせて学習率を変化させるscheduler（学習率スケジューラ）の使い方を身につける．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
import copy

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットとネットワークの準備
MNISTデータセットと畳み込みニューラルネットワーク（CNN）を用意します．

In [ ]:
train_data = torchvision.datasets.MNIST(root="./", train=True, transform=transforms.ToTensor(), download=True)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=100, shuffle=True)

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1)
        self.l1 = nn.Linear(14*14*4, 10)
        self.act = nn.Sigmoid()
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        h = self.pool(self.act(self.conv1(x)))
        h = h.view(h.size()[0], -1)
        h = self.l1(h)
        return h

## Optimizerの種類と特徴

ネットワークの学習では，誤差関数の勾配を用いてパラメータを更新するOptimizerを選択する必要があります．代表的なOptimizerには，以下のような違いがあります．

| Optimizer | 特徴 |
| :-- | :-- |
| SGD + Momentum | 勾配に加えて，過去の更新方向を慣性（momentum）として利用する．学習率などのハイパーパラメータの調整は必要だが，汎化性能が高くなりやすいとされる． |
| RMSprop | パラメータごとに，勾配の二乗の移動平均を用いて学習率を自動的に調整する．勾配のスケールが大きく変化する問題に強い． |
| Adam | Momentumの発想とRMSpropの発想を組み合わせた手法．デフォルトに近い設定でも収束が速く，扱いやすいためよく利用される． |
| AdamW | Adamにおけるweight decay（重みの正則化）の実装方法を改善した手法．L2正則化と勾配更新を分離しており，Transformerを用いたモデルなどで広く使われている． |

いずれもPyTorchでは`torch.optim`以下のクラスとして提供されており，`optimizer = torch.optim.Adam(model.parameters(), lr=0.001)`のように，ネットワークのパラメータと学習率などのハイパーパラメータを渡してインスタンスを作成します．

## 複数のOptimizerの比較

同じネットワーク構造・同じ初期パラメータから学習を開始し，Optimizerの種類だけを変えて学習した場合のlossの推移を比較します．
初期パラメータを揃えるため，1つのネットワークの`state_dict`を保存しておき，Optimizerごとに新しく作成したネットワークへ読み込んでから学習を行います．

In [ ]:
### 初期パラメータを揃えるため，1つのネットワークのstate_dictを保存しておく
base_model = CNN()
init_state_dict = copy.deepcopy(base_model.state_dict())

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

epoch_num = 3

optimizer_configs = {
    'SGD+Momentum': lambda params: torch.optim.SGD(params, lr=0.01, momentum=0.9),
    'RMSprop': lambda params: torch.optim.RMSprop(params, lr=0.001),
    'Adam': lambda params: torch.optim.Adam(params, lr=0.001),
    'AdamW': lambda params: torch.optim.AdamW(params, lr=0.001),
}

loss_history = {}

for name, make_optimizer in optimizer_configs.items():
    model = CNN()
    model.load_state_dict(init_state_dict)
    model = model.to(device)
    model.train()

    optimizer = make_optimizer(model.parameters())

    losses = []
    for epoch in range(epoch_num):
        for image, label in train_loader:
            image = image.to(device)
            label = label.to(device)

            y = model(image)
            loss = criterion(y, label)

            model.zero_grad()
            loss.backward()
            optimizer.step()

            losses.append(loss.item())

    loss_history[name] = losses
    print("{}: final loss = {:.4f}".format(name, losses[-1]))

In [ ]:
plt.figure(figsize=(8, 5))
for name, losses in loss_history.items():
    plt.plot(losses, label=name)
plt.xlabel('iteration')
plt.ylabel('loss')
plt.legend()
plt.title('Optimizerごとのlossの推移')
plt.show()

## Schedulerによる学習率の調整

Optimizerの学習率（`lr`）は固定値のまま学習を進めることもできますが，一般に学習の序盤は大きな学習率で大まかにパラメータを更新し，学習が進むにつれて学習率を小さくして細かく調整する方が，安定して良い精度が得られやすいことが知られています．
このように，学習の進行にあわせて学習率を変化させる仕組みがscheduler（学習率スケジューラ）です．

PyTorchでは`torch.optim.lr_scheduler`以下に，以下のような代表的なschedulerが用意されています．

| Scheduler | 特徴 |
| :-- | :-- |
| `StepLR` | 指定したエポック数（`step_size`）ごとに，学習率を`gamma`倍する． |
| `MultiStepLR` | 指定したエポック数のリスト（`milestones`）に到達するたびに，学習率を`gamma`倍する． |
| `ExponentialLR` | 毎エポック，学習率を`gamma`倍し，指数関数的に減衰させる． |
| `CosineAnnealingLR` | コサイン関数に従って，学習率を滑らかに減衰させる． |
| `ReduceLROnPlateau` | 検証データでのlossや精度が一定エポック改善しなかった場合に，学習率を`factor`倍する． |

`StepLR`や`CosineAnnealingLR`などは事前に決めたスケジュールに従って学習率を変化させるのに対し，`ReduceLROnPlateau`は検証データでの評価結果に応じて動的に学習率を変化させる点が異なります．

### Schedulerを用いた学習

`scheduler`は，作成した`optimizer`を渡して初期化し，通常は1エポックの学習が終わるたびに`scheduler.step()`を呼び出すことで，内部で管理している学習率を更新します．
ここでは，`StepLR`を用いて，2エポックごとに学習率を0.5倍にしながら学習を行います．
`optimizer.param_groups[0]['lr']`で，現在のOptimizerに設定されている学習率を確認できます．

In [ ]:
model = CNN()
model.load_state_dict(init_state_dict)
model = model.to(device)
model.train()

optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

epoch_num = 8
lr_history = []
loss_history_sched = []

for epoch in range(1, epoch_num+1):
    sum_loss = 0.0
    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)
        loss = criterion(y, label)

        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()

    current_lr = optimizer.param_groups[0]['lr']
    lr_history.append(current_lr)
    loss_history_sched.append(sum_loss / len(train_loader))
    print("epoch: {}, lr: {}, mean loss: {:.4f}".format(epoch, current_lr, sum_loss / len(train_loader)))

    ### 1エポックの学習が終わったタイミングでschedulerを更新
    scheduler.step()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(range(1, epoch_num+1), lr_history, marker='o')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('learning rate')
axes[0].set_title('学習率の推移（StepLR）')

axes[1].plot(range(1, epoch_num+1), loss_history_sched, marker='o')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('mean loss')
axes[1].set_title('lossの推移')

plt.tight_layout()
plt.show()

### Warmup付きCosine Annealing LR Scheduler

学習の開始直後は，ネットワークのパラメータがランダムな初期値のままであるため，勾配が不安定になりやすく，特にAdamなどの適応的なOptimizerや大きなバッチサイズを用いる場合，最初から大きな学習率で更新すると学習がうまく進まないことがあります．

そこで，学習開始から数エポック（あるいは数イテレーション）の間は，学習率を小さい値から目標の学習率まで徐々に線形に引き上げる**Warmup**を行い，その後`CosineAnnealingLR`のように学習率を滑らかに減衰させていく手法がよく用いられます．この「Warmup + Cosine Annealing」は，Transformerを用いたモデルの学習などで広く使われている代表的な学習率スケジュールです．

PyTorchには，WarmupとCosine Annealingを組み合わせた単体のschedulerは用意されていませんが，`torch.optim.lr_scheduler.SequentialLR`を使うことで，複数のschedulerを指定したエポックで順番に切り替えて利用できます．
Warmup部分には，学習率を`start_factor`倍から`end_factor`倍まで線形に変化させる`LinearLR`を用います．

In [ ]:
model = CNN()
model.load_state_dict(init_state_dict)
model = model.to(device)
model.train()

optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

warmup_epochs = 2
epoch_num = 8

### 学習率を0.1倍（0.01 * lr）から1.0倍（lr）まで，warmup_epochsかけて線形に引き上げる
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_epochs)
### warmup終了後の残りエポック数をかけて，学習率をコサイン関数にしたがって減衰させる
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epoch_num - warmup_epochs)

### warmup_epochsまではwarmup_scheduler，それ以降はcosine_schedulerを使用する
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_epochs]
)

lr_history_warmup = []
loss_history_warmup = []

for epoch in range(1, epoch_num+1):
    sum_loss = 0.0
    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)
        loss = criterion(y, label)

        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()

    current_lr = optimizer.param_groups[0]['lr']
    lr_history_warmup.append(current_lr)
    loss_history_warmup.append(sum_loss / len(train_loader))
    print("epoch: {}, lr: {:.5f}, mean loss: {:.4f}".format(epoch, current_lr, sum_loss / len(train_loader)))

    scheduler.step()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(range(1, epoch_num+1), lr_history_warmup, marker='o')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('learning rate')
axes[0].set_title('学習率の推移（Warmup + CosineAnnealingLR）')

axes[1].plot(range(1, epoch_num+1), loss_history_warmup, marker='o')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('mean loss')
axes[1].set_title('lossの推移')

plt.tight_layout()
plt.show()

学習率が最初の`warmup_epochs`の間は小さい値から目標の学習率まで線形に増加し，その後はコサイン関数にしたがって滑らかに減衰していく様子が確認できます．

## 課題

1. `StepLR`以外のscheduler（例えば`CosineAnnealingLR`や`ReduceLROnPlateau`）に変更し，学習率とlossの推移がどのように変化するか比較してみましょう．

    ヒント：`ReduceLROnPlateau`は他のschedulerと異なり，`scheduler.step(検証lossなどの値)`のように監視対象の値を引数として渡す必要があります．

2. Optimizerの初期学習率（`lr`）を変更して，各Optimizerの学習の安定性や収束の速さがどのように変化するか確認してみましょう．

3. `warmup_epochs`の値を変更し，Warmupの長さが学習率の推移やlossの推移にどのような影響を与えるか確認してみましょう．